<a href="https://colab.research.google.com/github/IshikaGeed/HackOweek/blob/main/Hack_o_Week7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
!pip install pandas numpy scikit-learn gradio plotly

In [24]:
import pandas as pd
import numpy as np

df = pd.read_csv("PJMW_hourly.csv")

df["Datetime"] = pd.to_datetime(df["Datetime"])

df = df.sort_values("Datetime")

df.set_index("Datetime", inplace=True)

df.head()

,PJMW_MW
Datetime,
2002-04-01 01:00:00,4374.0
2002-04-01 02:00:00,4306.0
2002-04-01 03:00:00,4322.0
2002-04-01 04:00:00,4359.0
2002-04-01 05:00:00,4436.0


In [25]:
df["hour"] = df.index.hour
df["day_of_week"] = df.index.dayofweek

df["is_weekend"] = df["day_of_week"].apply(lambda x: 1 if x >= 5 else 0)

df.head()

,PJMW_MW,hour,day_of_week,is_weekend
Datetime,,,,
2002-04-01 01:00:00,4374.0,1,0,0
2002-04-01 02:00:00,4306.0,2,0,0
2002-04-01 03:00:00,4322.0,3,0,0
2002-04-01 04:00:00,4359.0,4,0,0
2002-04-01 05:00:00,4436.0,5,0,0


In [26]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

features = df[["PJMW_MW","hour","day_of_week","is_weekend"]]

scaler = StandardScaler()
scaled = scaler.fit_transform(features)

kmeans = KMeans(n_clusters=3, random_state=42)

df["cluster"] = kmeans.fit_predict(scaled)

df.head()

,PJMW_MW,hour,day_of_week,is_weekend,cluster
Datetime,,,,,
2002-04-01 01:00:00,4374.0,1,0,0,0
2002-04-01 02:00:00,4306.0,2,0,0,0
2002-04-01 03:00:00,4322.0,3,0,0,0
2002-04-01 04:00:00,4359.0,4,0,0,0
2002-04-01 05:00:00,4436.0,5,0,0,0


In [27]:
weekday_usage = df[df["is_weekend"]==0]["PJMW_MW"].mean()
weekend_usage = df[df["is_weekend"]==1]["PJMW_MW"].mean()

dip = weekday_usage - weekend_usage

print("Average Weekday Usage:", round(weekday_usage,2))
print("Average Weekend Usage:", round(weekend_usage,2))
print("Weekend Energy Dip:", round(dip,2))

Average Weekday Usage: 5754.35
Average Weekend Usage: 5221.86
Weekend Energy Dip: 532.48


In [28]:
!pip install gradio

In [29]:
import gradio as gr

def energy_dashboard(hour):

    sample = df.iloc[int(hour)]

    return {
        "Datetime": str(sample.name),
        "Energy Usage (MW)": float(sample["PJMW_MW"]),
        "Cluster": int(sample["cluster"]),
        "Weekend": int(sample["is_weekend"])
    }

slider = gr.Slider(0, len(df)-1, step=1)

interface = gr.Interface(
    fn=energy_dashboard,
    inputs=slider,
    outputs="json",
    title="Admin Building Weekend Energy Analysis Dashboard"
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://efeaab8dea8ce2266b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
